<a href="https://colab.research.google.com/github/QingfangLiu/DS_learning/blob/main/my_gpt_coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tiktoken
import torch
import torch.nn as nn

### specify configuration of small GPT-2 model

In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # total number of unique tokens in the vocabulary
    "context_length": 1024, # max number of tokens the model can process as input as one time
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [7]:
# define layer normalization function
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)) # scale is learnable parameter (aka. gamma)
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # shift is learnable parameter (aka. beta)

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True) # dim=-1 means to calculate across the last dim (typically embedding dimension)
        var = x.var(dim=-1, keepdim=True, unbiased=False) # unbiased=False means calculating using biased estimator (n instead of n-1)
        norm_x = (x - mean)/torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [8]:
# define GELU activation function (a nonlinear function similar to relu but not quite the same)
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [9]:
# define a feedforward network using GELU activation function
# input dim: batch size * number of tokens * embedding size
# output has the same dimension (so can be easily stacked)
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
              nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # expands embedding dim by 4
              GELU(),
              nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # contracts the dim by 4 to return to original embedding dim
        )

    def forward(self, x):
        return self.layers(x)

In [10]:
# define multi-head attention (from chapter 3)
# the efficient way of implementing multi-head
# this becomes a core component of the transformer block
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        # usually d_in = d_out = emb_dim
        # each head gets head_dim to work with
        super().__init__()
        assert (d_out % num_heads == 0),\
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # gives an integer (not a float from /)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # shared bias parameter, set to be false in this notebook
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out) # a final linear projection
        self.dropout = nn.Dropout(dropout)
        self.register_buffer( # add a non-learnable tensor (not updated by gradients)
            "mask", # saved as self.mask
            torch.triu(torch.ones(context_length, context_length), # static mask to handle any sequence up to context_length
                       diagonal=1) # create a square matrix of 1 and only keep the upper triangle with the rest being zeros
        )

    def forward(self, x):
        # input x dimension: batch size, num_tokens, emb_dim
        # output dimension: batch size, num_tokens, emb_dim
        # no change in dimension, but the meaning is changed from embedding vector to context vector
        
        b, num_tokens, d_in = x.shape # x shape: batch size, num_tokens, d_in (d_in usually equals to emb_dim)
        # apply linear layers operates on the last dimension (i.e. transforms per token per batch)
        keys = self.W_key(x) # → shape: [b, num_tokens, d_out]
        queries = self.W_query(x) # → shape: [b, num_tokens, d_out]
        values = self.W_value(x) # → shape: [b, num_tokens, d_out]

        # reshape to allow multi-head operations
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # rearrange to batch, num_heads, num_tokens, head_dim
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # compute attention scores via dot products
        # multiplication is repeated for each batch and each head
        # in dim: batch, num_heads, num_tokens, num_tokens
        attn_scores = queries @ keys.transpose(2,3)

        # generate mask for causal attention
        # shape: num_tokens, num_tokens, upper triangular
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # when num_tokens < context_length, slicing out the full mask

        # set mask position to -inf
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # softmax over last dimension to get attention weights
        # attn_weights: [batch, num_heads, num_tokens, num_tokens]
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1 # scaled by head_dim
        )
        attn_weights = self.dropout(attn_weights) # apply dropout to attention weights

        # [batch, num_tokens, num_heads, head_dim]
        context_vec = (attn_weights @ values).transpose(1,2)

        # merge last two dims to [batch, num_tokens, d_out]
        context_vec = context_vec.contiguous().view( # contiguous() ensures flattening not breaking caused by earlier .transpose()
            b, num_tokens, self.d_out
        )

        # a final linear projection to mix information across heads
        context_vec = self.out_proj(context_vec)
        return context_vec


In [11]:
# define transformer block using multi-head attention and feedforward
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"]) # to apply before attention block
        self.norm2 = LayerNorm(cfg["emb_dim"]) # to apply before ff block (this has to be specified separately)
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x): # an attention block and a feedforward block

        shortcut = x
        x = self.norm1(x) # pre-norm is widely used in modern architectures
        x = self.att(x) # apply multi-head attention
        x = self.drop_shortcut(x) # apply dropout
        x = x + shortcut # add residual (shortcut connection, or skip connection)

        shortcut = x
        x = self.norm2(x) # still pre-norm
        x = self.ff(x) # apply feedforward layer
        x = self.drop_shortcut(x) # apply dropout
        x = x + shortcut # add residual
        return x

In [12]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) # initializes token embedding lookup table with random values
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"]) # initializes position embedding lookup table with random values
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg)
              for _ in range(cfg["n_layers"])] # a series of transformer layers (make lists, then * to unpack into arguments, then stack them)
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear( # a linear layer to transform from emb_dim to vocab_size for next token prediction
            cfg["emb_dim"], cfg["vocab_size"], bias=False # do not use a bias term
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape # batch size: # of sequences; seq_len: how many tokens in each sequence
        tok_embeds = self.tok_emb(in_idx) # tok_embeds: batch size * sequence length * emb_dim (adds one dimension to the input)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device) # a tensor of integers representing locations, output is seq_len * emb_dim
        )
        x = tok_embeds + pos_embeds # add via broadcasting, output dim: batch size * sequence length * emb_dim
        x = self.drop_emb(x)  # dropout before transformer
        x = self.trf_blocks(x) # transformer blocks
        x = self.final_norm(x) # layer normalization
        logits = self.out_head(x) # finaly linear layer to output logits
        return logits


### Run GPT model

In [14]:
# initializes a tokenizer object for the gpt2 model
tokenizer = tiktoken.get_encoding("gpt2")

# two text inputs (with the same token length)
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch = []
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [13]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
logits = model(batch)
logits.shape

torch.Size([2, 4, 50257])

### a function to generate text from GPTmodel output (autoregressive generation)

In [19]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
  # idx: [batch size, token size] 
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:] # keep only last context_size tokens if the sequence gets too long
        with torch.no_grad(): # no need to calculate gradients
            logits = model(idx_cond)

        logits = logits[:, -1, :] # focus on last token: batch size * vocab size
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) # dim: batch size
        idx = torch.cat((idx, idx_next), dim=1) # concatenate generated token to original - update idx
    return idx

In [16]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0) # unsqueeze creates a batch dimension
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [17]:
model.eval() # put model to evaluation mode (disables dropout)
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
Output length: 10


In [18]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist()) # decode function works on list
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue


## Model training

In [20]:
# revise the parameter settings
# reduce context length from 1024 to 256
GPT_CONFIG_124M = {
    "vocab_size": 50257, # total number of unique tokens in the vocabulary
    "context_length": 256, # max number of tokens the model can process as input as one time
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [21]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [22]:
# create utility functions for text to token ID conversion
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [23]:
start_context = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


### Data Loading and Sequential Train/Validation Split

In [40]:
# load the verdict text file
filepath = "the-verdict.txt"
with open(filepath, "r", encoding="utf-8") as file:
    text_data = file.read()
print("Characters:", len(text_data))
print("Tokens:", len(tokenizer.encode(text_data)))

Characters: 20479
Tokens: 5145


In [41]:
# split training and validation datasets
# a sequential split (rather than random split)
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [42]:
print("Characters in train_data:", len(train_data))
print("Tokens in train_data:", len(tokenizer.encode(train_data)))

Characters in train_data: 18431
Tokens in train_data: 4612


In [43]:
print("Characters in val_data:", len(val_data))
print("Tokens in val_data:", len(tokenizer.encode(val_data)))

Characters in val_data: 2048
Tokens in val_data: 534


### Defining GPT-Style Dataset and Dataloader for Token Chunks

In [28]:
# next need to use functions from chapter 2
import torch
from torch.utils.data import Dataset, DataLoader

In [33]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        # txt: full text string
        # tokenizer: convert text to list of token IDs
        # max_length: how many tokens per training example
        # stride: how far to shift the window each time
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        # break down long tokenized text into sliding input-output pairs for next token prediction
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length] # length of max_length
            output_chunk = token_ids[i + 1: i + max_length + 1] # shifted by 1 for next-token prediction
            self.input_ids.append(torch.tensor(input_chunk)) # create as a list (DataLoader will convert to batches automatically)
            self.target_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # dataset has lists of input and output pairs
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

In [30]:
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0,
)

In [31]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nVal loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Val loader:
torch.Size([2, 256]) torch.Size([2, 256])


### Evaluating Loss on Training and Validation Sets

In [49]:
def calc_loss_batch(input_batch, target_batch, model, device): # compute loss for a single batch
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

In [50]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None: # used to speed up evaluation in model training
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.987583584255642
Validation loss: 10.98110580444336


### Training an LLM

In [ ]:
# page 147-167